# Mini Project — India COVID-19 EDA: A Different Perspective

Same India COVID-19 dataset as the reference project, but with a different analysis focus.

**Focus:** Active cases, 7-day moving average, peak days, state growth speed, monthly patterns, death burden, recovery vs active burden, and vaccination rollout.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110


## 1. Load Dataset

In [ ]:
cases = pd.read_csv(
    "https://raw.githubusercontent.com/imdevskp/covid-19-india-data/master/complete.csv"
)

vax = pd.read_csv(
    "https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/vaccinations/country_data/India.csv"
)

print("Case data:", cases.shape)
print("Vaccination data:", vax.shape)
cases.head()


## 2. Inspect and Clean Data

In [ ]:
cases.columns = [
    "date","state","lat","long","confirmed",
    "deaths","cured","new_cases","new_deaths","new_recovered"
]

cases["date"] = pd.to_datetime(cases["date"])

num_cols = [
    "confirmed","deaths","cured",
    "new_cases","new_deaths","new_recovered"
]

for col in num_cols:
    cases[col] = pd.to_numeric(cases[col], errors="coerce").fillna(0)

name_fix = {
    "Telangana***":"Telangana",
    "Telengana":"Telangana",
    "Union Territory of Jammu and Kashmir":"Jammu and Kashmir",
    "Union Territory of Ladakh":"Ladakh",
    "Union Territory of Chandigarh":"Chandigarh"
}

cases["state"] = cases["state"].replace(name_fix)
cases = cases.drop_duplicates(["date","state"])
cases = cases.sort_values(["state","date"]).reset_index(drop=True)

cases["active"] = cases["confirmed"] - cases["deaths"] - cases["cured"]
cases["month"] = cases["date"].dt.to_period("M").astype(str)

print("Cleaned shape:", cases.shape)
print("Date range:", cases["date"].min().date(), "to", cases["date"].max().date())


## 3. National Daily Summary

In [ ]:
national = cases.groupby("date", as_index=False).agg(
    confirmed=("confirmed","sum"),
    deaths=("deaths","sum"),
    cured=("cured","sum"),
    new_cases=("new_cases","sum"),
    new_deaths=("new_deaths","sum"),
    new_recovered=("new_recovered","sum")
)

national["active"] = national["confirmed"] - national["deaths"] - national["cured"]

national["recovery_rate"] = np.where(
    national["confirmed"] > 0,
    national["cured"] / national["confirmed"] * 100,
    0
)

national["new_cases_7day_avg"] = (
    national["new_cases"].rolling(7, min_periods=1).mean()
)

national.tail()


## 4. Active Cases Over Time

In [ ]:
plt.figure(figsize=(9,5))
plt.plot(national["date"], national["active"], linewidth=2)
plt.title("India: Active COVID-19 Cases Over Time")
plt.xlabel("Date")
plt.ylabel("Active Cases")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 5. 7-Day Moving Average of New Cases

In [ ]:
plt.figure(figsize=(9,5))
plt.plot(national["date"], national["new_cases"], alpha=0.35, label="Daily New Cases")
plt.plot(national["date"], national["new_cases_7day_avg"], linewidth=2.5, label="7-Day Average")
plt.title("Daily Cases with 7-Day Moving Average")
plt.xlabel("Date")
plt.ylabel("New Cases")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 6. Peak-Day Analysis

In [ ]:
peak = national.loc[national["new_cases"].idxmax()]

print("Peak date:", peak["date"].date())
print("Peak new cases:", int(peak["new_cases"]))
print("Active cases on peak date:", int(peak["active"]))

top_10_days = national.nlargest(10, "new_cases")
top_10_days[["date","new_cases","new_deaths","new_recovered"]]


## 7. States with Fastest Average Case Accumulation

In [ ]:
first_positive = (
    cases[cases["confirmed"] > 0]
    .groupby("state")["date"]
    .min()
    .rename("first_positive_date")
)

latest_date = cases["date"].max()

state_latest = (
    cases[cases["date"] == latest_date]
    [["state","confirmed","deaths","cured","active"]]
    .set_index("state")
)

growth = state_latest.join(first_positive).reset_index()

growth["days_since_first_case"] = (
    latest_date - growth["first_positive_date"]
).dt.days.clip(lower=1)

growth["avg_cases_per_day"] = (
    growth["confirmed"] / growth["days_since_first_case"]
)

growth = growth.sort_values("avg_cases_per_day", ascending=False)

growth.head(10)


## 8. Highest Active-Case Burden by State

In [ ]:
active_states = (
    state_latest.sort_values("active", ascending=False)
    .head(12)
    .reset_index()
)

plt.figure(figsize=(9,6))
sns.barplot(
    data=active_states,
    y="state",
    x="active",
    hue="state",
    legend=False
)
plt.title("States with Highest Active-Case Burden")
plt.xlabel("Active Cases")
plt.ylabel("State / UT")
plt.tight_layout()
plt.show()


## 9. Monthly State-Level Heatmap

In [ ]:
monthly = cases.pivot_table(
    index="state",
    columns="month",
    values="new_cases",
    aggfunc="sum",
    fill_value=0
)

top_states = monthly.sum(axis=1).nlargest(15).index
heat_data = monthly.loc[top_states]

plt.figure(figsize=(11,7))
sns.heatmap(heat_data, cmap="YlOrRd", linewidths=0.2)
plt.title("Monthly New Cases — Top 15 States / UTs")
plt.xlabel("Month")
plt.ylabel("State / UT")
plt.tight_layout()
plt.show()


## 10. Monthly National Growth

In [ ]:
monthly_national = (
    national.assign(month=national["date"].dt.to_period("M").astype(str))
    .groupby("month", as_index=False)
    .agg(
        new_cases=("new_cases","sum"),
        new_deaths=("new_deaths","sum"),
        new_recovered=("new_recovered","sum")
    )
)

plt.figure(figsize=(10,5))
plt.bar(monthly_national["month"], monthly_national["new_cases"])
plt.title("Total New COVID-19 Cases by Month")
plt.xlabel("Month")
plt.ylabel("New Cases")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 11. Death Burden by State

In [ ]:
death_states = (
    state_latest.sort_values("deaths", ascending=False)
    .head(12)
    .reset_index()
)

plt.figure(figsize=(9,6))
sns.barplot(
    data=death_states,
    y="state",
    x="deaths",
    hue="state",
    legend=False
)
plt.title("States with Highest Reported COVID-19 Deaths")
plt.xlabel("Deaths")
plt.ylabel("State / UT")
plt.tight_layout()
plt.show()


## 12. Recovery Rate vs Active-Case Rate

In [ ]:
metrics = state_latest.copy()

metrics["recovery_rate"] = np.where(
    metrics["confirmed"] > 0,
    metrics["cured"] / metrics["confirmed"] * 100,
    0
)

metrics["active_rate"] = np.where(
    metrics["confirmed"] > 0,
    metrics["active"] / metrics["confirmed"] * 100,
    0
)

plot_data = metrics[metrics["confirmed"] >= 1000].reset_index()

plt.figure(figsize=(9,6))
sns.scatterplot(
    data=plot_data,
    x="recovery_rate",
    y="active_rate",
    size="confirmed",
    sizes=(40,600),
    alpha=0.7,
    legend=False
)

for _, row in plot_data.nlargest(8, "confirmed").iterrows():
    plt.annotate(
        row["state"],
        (row["recovery_rate"], row["active_rate"]),
        xytext=(4,4),
        textcoords="offset points",
        fontsize=8
    )

plt.title("Recovery Rate vs Active-Case Rate")
plt.xlabel("Recovery Rate (%)")
plt.ylabel("Active Cases as % of Confirmed")
plt.tight_layout()
plt.show()


## 13. Vaccination Rollout

In [ ]:
vax["date"] = pd.to_datetime(vax["date"])
vax = vax.sort_values("date")

vax["daily_doses"] = vax["total_vaccinations"].diff()
vax["daily_doses_7day_avg"] = (
    vax["daily_doses"].rolling(7, min_periods=1).mean()
)

plt.figure(figsize=(9,5))
plt.plot(
    vax["date"],
    vax["total_vaccinations"] / 1e9,
    linewidth=2
)
plt.title("India: Cumulative COVID-19 Vaccination Progress")
plt.xlabel("Date")
plt.ylabel("Total Doses (Billion)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 14. Vaccination Speed

In [ ]:
plt.figure(figsize=(9,5))
plt.plot(
    vax["date"],
    vax["daily_doses_7day_avg"] / 1e6,
    linewidth=2
)
plt.title("India: 7-Day Average Daily Vaccination Speed")
plt.xlabel("Date")
plt.ylabel("Doses per Day (Million)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 15. Key Findings

In [ ]:
print("FINAL NATIONAL SNAPSHOT")
print(national.tail(1).T)

print("\nTOP STATES BY ACTIVE CASES")
print(state_latest.nlargest(10, "active")[["confirmed","active"]])

print("\nFASTEST AVERAGE CASE ACCUMULATION")
print(growth.head(10)[
    ["state","confirmed","days_since_first_case","avg_cases_per_day"]
])

print("\nPEAK NEW-CASE DAY")
print(peak[["date","new_cases","active"]])


## 16. Conclusion

This project uses the same India COVID-19 dataset as the reference project but studies it from a different perspective.

The main focus is on active-case burden, smoothed daily trends, peak days, state growth speed, monthly patterns, death burden, recovery efficiency and vaccination speed.

The project shows that one dataset can be explored in different ways by creating new features and asking different analytical questions.


## 17. Export Cleaned Dataset

In [ ]:
cases.to_csv("Arnav_cleaned_covid_india.csv", index=False)
print("Cleaned dataset exported successfully.")
